### LLM Chain 만들기


## 1. 환경 구성

### 1) 라이브러리 설치

In [ ]:
# uv add python-dotenv langchain langchain-openai

### 2) OpenAI 인증키 설정
https://openai.com/

In [1]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 값은 출력하지 않고 설정 여부만 확인
if not OPENAI_API_KEY:
    raise RuntimeError(".env 파일에 OPENAI_API_KEY 가 없습니다.")
print("OPENAI_API_KEY 설정 확인 완료")

OPENAI_API_KEY 설정 확인 완료


In [2]:
import langchain
print(langchain.__version__)

1.4.3


In [6]:
from langchain_openai import ChatOpenAI

# OpenAI 공통 설정 — 모델/온도 변경은 이곳만 수정
OPENAI_MODEL = "gpt-4o-mini"  # 대안: "gpt-4o"
TEMPERATURE = 0.7


def get_llm(model: str = OPENAI_MODEL, temperature: float = TEMPERATURE) -> ChatOpenAI:
    """OpenAI API를 사용하는 ChatOpenAI 인스턴스를 생성합니다."""
    return ChatOpenAI(
        api_key=OPENAI_API_KEY,
        model=model,
        temperature=temperature,
    )


# 기본 llm (이후 셀에서 재사용)
llm = get_llm()

## 2. LLM Chain

### 1) Prompt + LLM


In [5]:
# model: 공통 설정 셀의 llm 사용

# chain 실행
result = llm.invoke("인공지능 모델의 학습 원리에 대하여 쉽게 설명해 주세요.")
print(type(result))
print(result)

NameError: name 'llm' is not defined

In [ ]:
print(result.content)

### 2) PromptTemplate + LLM

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("You are an expert in AI Expert. Answer the question.\
                                       <Question>: {input}에 대해 쉽게 설명해주세요.")

print(type(prompt))
print(prompt)

In [4]:
# llm: 공통 설정 셀의 llm 사용
# chain 연결 (LCEL)
chain = prompt | llm
print(type(chain))

# chain 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

NameError: name 'prompt' is not defined

In [ ]:
print(result.content)

### 3) PromptTemplate + LLM(invoke()) + StrOutputParser

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

output_parser = StrOutputParser()

# 2. chain 생성 (LCEL)
chain = prompt | llm | output_parser
print(type(chain))

# 3. chain의 invoke 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

### 4) PromptTemplate + LLM(stream()) + StrOutputParser

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

# chain 연결 (LCEL)
chain = prompt | llm | StrOutputParser()

# 스트리밍 출력을 위한 요청
answer = chain.stream({"input": "인공지능 모델의 학습 원리"})
# 스트리밍 출력
#print(answer)

for token in answer:
    # 스트림에서 받은 데이터의 내용을 출력합니다. 줄바꿈 없이 이어서 출력하고, 버퍼를 즉시 비웁니다.
    print(token, end="", flush=True)

인공지능 모델의 학습 원리는 기본적으로 데이터를 통해 패턴을 인식하고, 이를 바탕으로 예측이나 결정을 내리는 것입니다. 이를 쉽게 설명하자면, 다음과 같은 단계로 나눌 수 있습니다.

1. **데이터 수집**: 모델을 학습시키기 위해서는 먼저 많은 양의 데이터가 필요합니다. 이 데이터는 텍스트, 이미지, 음성 등 다양한 형태일 수 있습니다.

2. **전처리**: 수집한 데이터는 종종 노이즈가 포함되어 있거나 불완전합니다. 따라서 데이터를 정리하고, 필요한 형식으로 변환하는 과정을 거칩니다. 예를 들어, 텍스트 데이터에서 불필요한 단어를 제거하거나, 이미지를 일정한 크기로 맞추는 작업이 포함됩니다.

3. **모델 선택**: 학습에 사용할 인공지능 모델을 선택합니다. 예를 들어, 이미지 인식에는 CNN(합성곱 신경망), 자연어 처리에는 RNN(순환 신경망)이나 트랜스포머 모델이 자주 사용됩니다.

4. **학습**: 선택한 모델에 데이터를 입력하고, 모델이 데이터의 패턴을 학습하도록 합니다. 이 과정에서 모델은 입력 데이터에 대한 출력(예: 정답)을 비교하고, 그 차이를 줄이기 위해 가중치를 조정합니다. 이를 '역전파'라고 하며, 반복적으로 수행됩니다.

5. **검증 및 테스트**: 학습이 끝난 후, 모델의 성능을 평가하기 위해 별도의 검증 및 테스트 데이터를 사용합니다. 이를 통해 모델이 실제 데이터에서 얼마나 잘 작동하는지를 확인합니다.

6. **배포**: 모델의 성능이 만족스럽다면, 실제 애플리케이션에 배포하여 사용자에게 서비스를 제공합니다.

7. **모델 업데이트**: 시간이 지나면서 새로운 데이터가 생기므로, 모델을 주기적으로 업데이트하여 성능을 유지해야 합니다.

이러한 과정을 통해 인공지능 모델은 입력 데이터에 대한 이해도를 높이고, 새로운 데이터에 대해 더 나은 예측을 할 수 있게 됩니다.

##### 2) Multiple Chains
* Multi Chain을 활용한 영화 추천 및 줄거리 요약 

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고,\
                                            줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요")

# 영화 예제용 모델 (공통 설정 셀의 get_llm 사용)
llm = get_llm(OPENAI_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
movie = chain1.invoke({"genre": "Drama"})  # 영화 제목 얻기

print(type(movie))
print(" ===> 추천된 영화:")  # movie 값 출력
print(movie)


<class 'langchain_core.messages.base.TextAccessor'>
 ===> 추천된 영화:
드라마 장르에서 추천할 만한 영화로 **"쇼생크 탈출" (The Shawshank Redemption)**을 추천합니다. 이 영화는 두 주인공이 감옥에서 겪는 우정과 희망을 이야기하며, 인간의 의지와 자유에 대한 깊은 메시지를 전달합니다. 감정적으로도 깊이 있는 작품으로 많은 사랑을 받고 있습니다. 꼭 한 번 보시길 추천드립니다!


In [ ]:
# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)
print(chain2)

# 실행: "SF" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:\n", response) 

##### chain1과 chain2에서 영화 제목이 일관되게 전달 되도록 변경 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from pprint import pprint

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고, 줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요.")

llm = get_llm(OPENAI_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

# 실행: "Drama" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:")
pprint(response)

In [ ]:
# 모델 설정 (공통 설정 셀의 get_llm 사용)
model = get_llm(OPENAI_MODEL)

# 멀티 체인 실행 함수
def movie_chain(genre):
    print(f"\n🔹 1단계: '{genre}' 장르 영화 추천")
    
    # 체인 1: 영화 추천
    response1 = model.invoke(f"{genre} 장르에서 유명한 영화 제목 하나만 알려줘")
    movie = response1.content.strip()
    print(f"   → 추천 영화: {movie}")
    
    print(f"\n🔹 2단계: '{movie}' 영화 정보 조회")
    
    # 체인 2: 영화 정보
    response2 = model.invoke(f"{movie} 영화에 대해 간단히 알려줘. 줄거리, 감독, 출연진 포함해서")
    info = response2.content
    print(f"   → 영화 정보:\n{info}")
    
    return movie, info

# 실행
genre = input("원하는 장르를 입력하세요: ")
movie, info = movie_chain(genre)
print(f"\n 최종 결과: '{movie}' 영화가 추천되었습니다!")